# 02 - Trích Xuất Đặc Trưng IEMOCAP Full: Emotion2Vec + Acoustic

Notebook này tạo **full feature cache** cho model 03/04.

Output bắt buộc:

```text
data/features/iemocap_full_emotion2vec_acoustic_cache.npz
  sample_ids
  emotion2vec      # Branch A: frozen Emotion2Vec utterance embedding
  acoustic         # Branch B: handcrafted acoustic vector
  acoustic_names
```

Khác với bản baseline trước, notebook này không coi metadata là feature chính. Muốn train model full thì phải có cache này.

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
def zip_folder(source_dir, zip_path, exclude_dir_names=None, exclude_suffixes=None):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    exclude_dir_names = set(exclude_dir_names or [])
    exclude_suffixes = set(exclude_suffixes or [])
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in source_dir.rglob("*"):
            if not file_path.is_file():
                continue
            if any(part in exclude_dir_names for part in file_path.relative_to(source_dir).parts):
                continue
            if file_path.suffix.lower() in exclude_suffixes:
                continue
            zf.write(file_path, file_path.relative_to(source_dir))
    return zip_path

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "Speech Project" and PROJECT_ROOT.parent.name == "Speech Project":
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

BASE_DIR = PROJECT_ROOT / "06_w2v_based_models"
if not BASE_DIR.exists() and Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working") / "06_w2v_based_models"

LOCAL_DATA_DIR = PROJECT_ROOT / "06_w2v_based_models" / "data"

def has_train_tables(path):
    return (
        path is not None
        and (path / "metadata" / "iemocap_metadata_full.csv").exists()
        and (path / "splits").exists()
    )

def candidate_data_roots(base):
    if base is None:
        return []
    base = Path(base)
    roots = [base, base / "data"]
    if base.exists():
        for meta_path in base.rglob("iemocap_metadata_full.csv"):
            roots.append(meta_path.parent.parent)
    return roots

def resolve_data_dir():
    candidates = []
    env_dir = os.environ.get("IEMOCAP_DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir))
    candidates.extend([
        LOCAL_DATA_DIR,
        Path("/kaggle/input/iemocap-full-multitask-data"),
        Path("/kaggle/input/iemocap_full_multitask_data"),
        Path("/kaggle/input/iemocap-multitask-train-data"),
        Path("/kaggle/input/iemocap_multitask_train_data"),
    ])

    for candidate in candidates:
        for root in candidate_data_roots(candidate):
            if has_train_tables(root):
                return root.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for root in candidate_data_roots(kaggle_input):
            if has_train_tables(root):
                return root.resolve()

    raise FileNotFoundError(
        "Không tìm thấy dữ liệu IEMOCAP. Notebook sẽ tự quét /kaggle/input, nhưng folder dữ liệu cần có "
        "metadata/iemocap_metadata_full.csv và splits/. Nếu bạn đặt ở chỗ khác, set IEMOCAP_DATA_DIR."
    )

DATA_DIR = resolve_data_dir()
METADATA_DIR = DATA_DIR / "metadata"
SPLIT_DIR = DATA_DIR / "splits"
AUDIO_DIR = DATA_DIR / "audio_wav"

FULL_METADATA_PATH = METADATA_DIR / "iemocap_metadata_full.csv"

WORKING_DATA_DIR = Path("/kaggle/working/iemocap_full_multitask_data") if Path("/kaggle/working").exists() else DATA_DIR
WORKING_FEATURE_DIR = WORKING_DATA_DIR / "features"
WORKING_REPORT_DIR = WORKING_DATA_DIR / "feature_reports"
WORKING_FIGURE_DIR = WORKING_DATA_DIR / "feature_figures"
INPUT_FEATURE_DIR = DATA_DIR / "features"
FEATURE_CACHE_NAME = "iemocap_full_emotion2vec_acoustic_cache.npz"
WORKING_FEATURE_CACHE_PATH = WORKING_FEATURE_DIR / FEATURE_CACHE_NAME
INPUT_FEATURE_CACHE_PATH = INPUT_FEATURE_DIR / FEATURE_CACHE_NAME

def resolve_feature_cache_path(require_exists=False):
    env_cache = os.environ.get("IEMOCAP_FEATURE_CACHE", "").strip()
    candidates = []
    if env_cache:
        candidates.append(Path(env_cache))
    candidates.extend([WORKING_FEATURE_CACHE_PATH, INPUT_FEATURE_CACHE_PATH])
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    if require_exists:
        raise FileNotFoundError(
            "Không tìm thấy full feature cache. Hãy chạy notebook 02 trước. Trên Kaggle, notebook 02 ghi cache vào "
            f"{WORKING_FEATURE_CACHE_PATH}, không ghi vào /kaggle/input vì Kaggle input dataset là read-only."
        )
    return WORKING_FEATURE_CACHE_PATH

print("DATA_DIR:", DATA_DIR)
print("AUDIO_DIR:", AUDIO_DIR, AUDIO_DIR.exists())
print("INPUT_FEATURE_CACHE_PATH:", INPUT_FEATURE_CACHE_PATH, INPUT_FEATURE_CACHE_PATH.exists())
print("WORKING_FEATURE_CACHE_PATH:", WORKING_FEATURE_CACHE_PATH, WORKING_FEATURE_CACHE_PATH.exists())
print("WORKING_REPORT_DIR:", WORKING_REPORT_DIR)
print("WORKING_FIGURE_DIR:", WORKING_FIGURE_DIR)

In [ ]:
TARGET_SR = 16000
MAX_FILES = os.getenv("MAX_FILES", "").strip()
MAX_FILES = int(MAX_FILES) if MAX_FILES else None
EMOTION2VEC_MODEL_NAME = os.getenv("EMOTION2VEC_MODEL", "iic/emotion2vec_base")
IS_KAGGLE = Path("/kaggle/working").exists()
INSTALL_DEPS = os.getenv("INSTALL_EMOTION2VEC_DEPS", "1" if IS_KAGGLE else "0") == "1"

def maybe_install_deps():
    if not INSTALL_DEPS:
        return
    import importlib.util
    import subprocess
    import sys
    missing = [pkg for pkg in ["funasr", "modelscope", "librosa", "soundfile"] if importlib.util.find_spec(pkg) is None]
    if missing:
        print("Đang cài package còn thiếu cho Emotion2Vec:", missing)
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "funasr", "modelscope", "librosa", "soundfile",
            "addict", "simplejson", "sortedcontainers",
        ])
    else:
        print("Các package Emotion2Vec đã có sẵn.")

def require_audio_dir():
    if not AUDIO_DIR.exists():
        raise FileNotFoundError(
            f"Không tìm thấy audio directory: {AUDIO_DIR}. Với mô hình full, Kaggle Dataset cần có audio_wav/, "
            "hoặc bạn phải upload sẵn feature cache vào features/."
        )

def load_audio_16k(path):
    import librosa
    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    if len(y) == 0:
        y = np.zeros(TARGET_SR, dtype=np.float32)
    peak = np.max(np.abs(y)) + 1e-8
    y = (y / peak).astype(np.float32)
    return y, TARGET_SR

def acoustic_vector(y, sr):
    import librosa
    feats = []
    names = []

    def add(name, arr):
        arr = np.asarray(arr, dtype=np.float32).reshape(-1)
        feats.extend([float(np.nanmean(arr)), float(np.nanstd(arr)), float(np.nanmin(arr)), float(np.nanmax(arr))])
        names.extend([f"{name}_mean", f"{name}_std", f"{name}_min", f"{name}_max"])

    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta = librosa.feature.delta(mfcc)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64), ref=np.max)

    add("rms", rms)
    add("zcr", zcr)
    add("centroid", centroid)
    add("bandwidth", bandwidth)
    add("rolloff", rolloff)
    for i in range(mfcc.shape[0]):
        add(f"mfcc_{i+1:02d}", mfcc[i])
        add(f"delta_mfcc_{i+1:02d}", delta[i])
    add("logmel", mel)

    return np.asarray(feats, dtype=np.float32), names

def load_emotion2vec_model():
    maybe_install_deps()
    try:
        from funasr import AutoModel
    except Exception as exc:
        raise ImportError(
            "Không import được funasr. Trên Kaggle hãy bật Internet. Notebook đã mặc định tự cài khi ở Kaggle; "
            "nếu bạn tắt cơ chế này thì set INSTALL_EMOTION2VEC_DEPS=1 rồi chạy lại cell."
        ) from exc
    print("Loading emotion2vec:", EMOTION2VEC_MODEL_NAME)
    return AutoModel(model=EMOTION2VEC_MODEL_NAME, hub="ms")

def first_numeric_array(obj):
    if isinstance(obj, np.ndarray) and np.issubdtype(obj.dtype, np.number):
        return obj
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().numpy()
    if isinstance(obj, dict):
        preferred = ["feats", "embedding", "embeddings", "hidden_states", "xvector", "scores"]
        for key in preferred:
            if key in obj:
                arr = first_numeric_array(obj[key])
                if arr is not None:
                    return arr
        for value in obj.values():
            arr = first_numeric_array(value)
            if arr is not None:
                return arr
    if isinstance(obj, (list, tuple)):
        for value in obj:
            arr = first_numeric_array(value)
            if arr is not None:
                return arr
    return None

def extract_emotion2vec_embedding(model, wav_path):
    result = model.generate(str(wav_path), granularity="utterance", extract_embedding=True)
    arr = first_numeric_array(result)
    if arr is None:
        raise ValueError(f"No numeric emotion2vec embedding returned for {wav_path}")
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 0:
        arr = arr.reshape(1)
    if arr.ndim > 1:
        arr = arr.reshape(-1, arr.shape[-1]).mean(axis=0)
    return arr.astype(np.float32)

def resolve_wav_path(row):
    name = Path(str(row.get("wav_path", row.get("source_filename", "")))).name
    direct = AUDIO_DIR / name
    if direct.exists():
        return direct
    original = Path(str(row.get("wav_path", "")))
    if original.exists():
        return original
    raise FileNotFoundError(f"Cannot find WAV for sample {row.get('train_sample_id')} ({name})")

In [ ]:
require_audio_dir()
WORKING_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
WORKING_REPORT_DIR.mkdir(parents=True, exist_ok=True)
WORKING_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
metadata = pd.read_csv(FULL_METADATA_PATH)
metadata = metadata[metadata["emotion_4class"].isin(["neutral", "angry", "sad", "happy"])].copy()
metadata = metadata.drop_duplicates("train_sample_id").reset_index(drop=True)
if MAX_FILES is not None:
    metadata = metadata.head(MAX_FILES).copy()
print("Rows to extract:", len(metadata))
display(metadata[["train_sample_id", "emotion_4class", "wav_path"]].head())

## 1. Bảng đặc trưng và vai trò trong mô hình

Phần này ghi rõ mỗi nhóm đặc trưng được trích như thế nào, phục vụ nhánh nào trong mô hình, và vì sao cần nó.

Điểm quan trọng của mô hình full:

- `Emotion2Vec` là **nhánh riêng**, nhận waveform 16 kHz và tạo embedding cảm xúc mức cao.
- Acoustic handcrafted feature là **nhánh riêng**, dùng để bổ sung các tín hiệu mà embedding pretrained có thể không biểu diễn rõ: năng lượng, texture phổ, MFCC dynamics.
- Hai nhánh không thay thế nhau; chúng được fusion bằng cross-attention trong notebook 03/04.

In [ ]:
feature_reference_rows = [
    {
        "feature_group": "Emotion2Vec embedding",
        "branch": "Branch A - pretrained emotion representation",
        "how_to_extract": "waveform 16 kHz -> frozen emotion2vec_base -> utterance embedding",
        "why_it_matters": "Cung cấp high-level emotion cue đã học trước từ speech emotion representation.",
        "used_by_model": "Emotion2Vec adapter MLP, cross-attention fusion, emotion/AVD heads",
        "reference": "emotion2vec: Self-Supervised Pre-Training for Speech Emotion Representation; ModelScope iic/emotion2vec_base",
    },
    {
        "feature_group": "MFCC",
        "branch": "Branch B - handcrafted acoustic",
        "how_to_extract": "waveform -> Mel spectrum -> cepstral coefficients",
        "why_it_matters": "Mô tả timbre/màu giọng và spectral envelope.",
        "used_by_model": "Acoustic vector statistics",
        "reference": "SER feature engineering; librosa.feature.mfcc",
    },
    {
        "feature_group": "Delta MFCC",
        "branch": "Branch B - handcrafted acoustic",
        "how_to_extract": "đạo hàm bậc 1 của MFCC theo thời gian",
        "why_it_matters": "Cho biết màu giọng thay đổi nhanh hay chậm.",
        "used_by_model": "Acoustic vector statistics",
        "reference": "Delta features for speech dynamics; librosa.feature.delta",
    },
    {
        "feature_group": "RMS energy",
        "branch": "Branch B - handcrafted acoustic",
        "how_to_extract": "root mean square energy theo frame",
        "why_it_matters": "Đo cường độ/loudness; hữu ích cho angry, happy, low-energy sad/neutral.",
        "used_by_model": "Acoustic vector statistics",
        "reference": "librosa.feature.rms",
    },
    {
        "feature_group": "ZCR",
        "branch": "Branch B - handcrafted acoustic",
        "how_to_extract": "zero-crossing rate theo frame",
        "why_it_matters": "Bổ sung cue về voiced/unvoiced, độ sắc, texture high-frequency.",
        "used_by_model": "Acoustic vector statistics",
        "reference": "librosa.feature.zero_crossing_rate",
    },
    {
        "feature_group": "Spectral centroid / bandwidth / rolloff",
        "branch": "Branch B - handcrafted acoustic",
        "how_to_extract": "thống kê hình dạng phổ theo frame",
        "why_it_matters": "Mô tả độ sáng, độ rộng phổ và mức năng lượng vùng tần số cao.",
        "used_by_model": "Acoustic vector statistics",
        "reference": "librosa spectral features; acoustic feature fusion SER papers",
    },
    {
        "feature_group": "Log-Mel spectrogram summary",
        "branch": "Branch B - handcrafted acoustic",
        "how_to_extract": "Mel spectrogram -> log scale -> mean/std/min/max",
        "why_it_matters": "Giữ thông tin time-frequency tổng quát, bổ sung cho MFCC.",
        "used_by_model": "Acoustic vector statistics",
        "reference": "log-Mel is standard for CNN/audio emotion recognition",
    },
]

feature_reference = pd.DataFrame(feature_reference_rows)
feature_reference.to_csv(WORKING_REPORT_DIR / "feature_reference_table.csv", index=False, encoding="utf-8-sig")
display(feature_reference)

## 2. Thống kê dữ liệu trước khi trích đặc trưng

Mục tiêu của phần này là kiểm tra lại dataset sau khi lọc 4 emotion chính. Nếu dữ liệu lệch lớp hoặc thiếu audio, phần train phía sau sẽ bị ảnh hưởng trực tiếp.

In [ ]:
overview = {
    "n_rows": len(metadata),
    "n_speakers": metadata["speaker_id"].nunique() if "speaker_id" in metadata.columns else np.nan,
    "n_sessions": metadata["session"].nunique() if "session" in metadata.columns else np.nan,
    "audio_dir": str(AUDIO_DIR),
    "audio_file_count": len(list(AUDIO_DIR.glob("*.wav"))) if AUDIO_DIR.exists() else 0,
}
display(pd.DataFrame([overview]))

emotion_counts = metadata["emotion_4class"].value_counts().rename_axis("emotion").reset_index(name="n")
emotion_counts["percent"] = emotion_counts["n"] / emotion_counts["n"].sum() * 100
emotion_counts.to_csv(WORKING_REPORT_DIR / "emotion_distribution.csv", index=False, encoding="utf-8-sig")
display(emotion_counts)

if "duration" in metadata.columns:
    duration_summary = metadata.groupby("emotion_4class")["duration"].agg(["count", "mean", "std", "min", "median", "max"]).reset_index()
    duration_summary.to_csv(WORKING_REPORT_DIR / "duration_by_emotion.csv", index=False, encoding="utf-8-sig")
    display(duration_summary)

if "session" in metadata.columns:
    session_emotion = pd.crosstab(metadata["session"], metadata["emotion_4class"])
    session_emotion.to_csv(WORKING_REPORT_DIR / "session_emotion_crosstab.csv", encoding="utf-8-sig")
    display(session_emotion)

## 3. Chuẩn hóa waveform trước khi trích đặc trưng

Mỗi audio được đưa về:

- mono
- 16 kHz
- biên độ chuẩn hóa theo peak

Lý do: Emotion2Vec, wav2vec2, HuBERT, WavLM đều nhận waveform/raw speech. MFCC/log-Mel/RMS/ZCR cũng phụ thuộc trực tiếp vào sample rate và biên độ, nên nếu không chuẩn hóa trước thì feature giữa các file sẽ khó so sánh.

In [ ]:
maybe_install_deps()

selected_samples = (
    metadata.sort_values(["emotion_4class", "train_sample_id"])
    .groupby("emotion_4class", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)
display_cols = [c for c in ["train_sample_id", "emotion_4class", "speaker_id", "session", "duration", "transcription"] if c in selected_samples.columns]
display(selected_samples[display_cols])

def visualize_sample(row, axes):
    import librosa
    wav_path = resolve_wav_path(row)
    y, sr = load_audio_16k(wav_path)
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64), ref=np.max)
    t = np.arange(len(y)) / sr
    frame_t = np.linspace(0, len(y) / sr, len(rms))

    axes[0].plot(t, y, linewidth=0.6)
    axes[0].set_title(f"{row['emotion_4class']} waveform")
    axes[0].set_xlabel("time (s)")

    axes[1].plot(frame_t, rms, label="RMS", linewidth=1.0)
    axes[1].plot(frame_t, zcr, label="ZCR", linewidth=1.0)
    axes[1].set_title("RMS / ZCR")
    axes[1].legend(fontsize=8)

    axes[2].imshow(mfcc, aspect="auto", origin="lower", interpolation="nearest")
    axes[2].set_title("MFCC")

    axes[3].imshow(mel, aspect="auto", origin="lower", interpolation="nearest")
    axes[3].set_title("log-Mel")

fig, axes = plt.subplots(len(selected_samples), 4, figsize=(18, 3.2 * len(selected_samples)))
if len(selected_samples) == 1:
    axes = np.expand_dims(axes, 0)
for row_idx, (_, row) in enumerate(selected_samples.iterrows()):
    visualize_sample(row, axes[row_idx])
plt.tight_layout()
fig_path = WORKING_FIGURE_DIR / "feature_visualization_one_sample_per_emotion.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print("Saved:", fig_path)

## 4. So sánh cùng câu nói hoặc các câu khác nhau

Nếu metadata có nhiều mẫu trùng transcript, notebook sẽ tìm một câu xuất hiện ở nhiều emotion/speaker để so sánh. Nếu không có transcript trùng, notebook vẫn tạo bảng so sánh các mẫu đại diện khác emotion.

In [ ]:
comparison_rows = []
if "transcription" in metadata.columns:
    tmp = metadata.dropna(subset=["transcription"]).copy()
    tmp["transcription_norm"] = tmp["transcription"].astype(str).str.lower().str.strip()
    candidate = (
        tmp.groupby("transcription_norm")
        .agg(n=("train_sample_id", "count"), n_emotions=("emotion_4class", "nunique"))
        .query("n >= 2")
        .sort_values(["n_emotions", "n"], ascending=False)
        .head(1)
    )
    if len(candidate):
        key = candidate.index[0]
        comparison_rows = tmp[tmp["transcription_norm"].eq(key)].groupby("emotion_4class", group_keys=False).head(1).head(4)
        display(Markdown(f"Transcript được chọn: `{key}`"))

if len(comparison_rows) == 0:
    comparison_rows = selected_samples.copy()
    display(Markdown("Không tìm thấy transcript trùng đủ tốt; dùng mỗi emotion một mẫu đại diện để so sánh."))

summary_rows = []
for _, row in comparison_rows.iterrows():
    wav_path = resolve_wav_path(row)
    y, sr = load_audio_16k(wav_path)
    a_vec, a_names = acoustic_vector(y, sr)
    vals = dict(zip(a_names, a_vec))
    summary_rows.append({
        "train_sample_id": row["train_sample_id"],
        "emotion": row["emotion_4class"],
        "speaker_id": row.get("speaker_id", ""),
        "duration": row.get("duration", np.nan),
        "rms_mean": vals.get("rms_mean", np.nan),
        "zcr_mean": vals.get("zcr_mean", np.nan),
        "centroid_mean": vals.get("centroid_mean", np.nan),
        "mfcc_01_mean": vals.get("mfcc_01_mean", np.nan),
        "logmel_mean": vals.get("logmel_mean", np.nan),
    })

comparison_df = pd.DataFrame(summary_rows)
comparison_df.to_csv(WORKING_REPORT_DIR / "selected_sample_acoustic_comparison.csv", index=False, encoding="utf-8-sig")
display(comparison_df)

## 5. Thống kê acoustic trên một subset trước khi extract full

Phần này không thay thế training feature cache. Nó chỉ giúp kiểm tra xem feature có giá trị hợp lý không, có bị toàn 0/NaN không, và các emotion có phân bố acoustic khác nhau ở mức mô tả hay không.

In [ ]:
ANALYSIS_SAMPLE_N = int(os.getenv("ACOUSTIC_ANALYSIS_SAMPLE_N", "160"))
per_emotion_n = max(1, ANALYSIS_SAMPLE_N // max(1, metadata["emotion_4class"].nunique()))
analysis_subset = (
    metadata.groupby("emotion_4class", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), per_emotion_n), random_state=SEED))
    .reset_index(drop=True)
)

rows = []
feature_names = None
for _, row in analysis_subset.iterrows():
    y, sr = load_audio_16k(resolve_wav_path(row))
    a_vec, a_names = acoustic_vector(y, sr)
    feature_names = a_names
    d = {"train_sample_id": row["train_sample_id"], "emotion": row["emotion_4class"]}
    d.update(dict(zip(a_names, a_vec)))
    rows.append(d)

acoustic_demo = pd.DataFrame(rows)
acoustic_demo.to_csv(WORKING_REPORT_DIR / "acoustic_demo_subset_features.csv", index=False, encoding="utf-8-sig")

focus_cols = [c for c in ["rms_mean", "zcr_mean", "centroid_mean", "bandwidth_mean", "rolloff_mean", "mfcc_01_mean", "logmel_mean"] if c in acoustic_demo.columns]
demo_summary = acoustic_demo.groupby("emotion")[focus_cols].agg(["mean", "std"]).round(5)
demo_summary.to_csv(WORKING_REPORT_DIR / "acoustic_demo_summary_by_emotion.csv", encoding="utf-8-sig")
display(demo_summary)

fig, axes = plt.subplots(1, len(focus_cols[:4]), figsize=(4.2 * len(focus_cols[:4]), 4))
if len(focus_cols[:4]) == 1:
    axes = [axes]
for ax, col in zip(axes, focus_cols[:4]):
    grouped = acoustic_demo.groupby("emotion")[col].mean().reindex(["neutral", "angry", "sad", "happy"])
    ax.bar(grouped.index, grouped.values)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
fig_path = WORKING_FIGURE_DIR / "acoustic_demo_feature_means_by_emotion.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print("Saved:", fig_path)

## 6. Extract full cache cho notebook 03/04

Cell này sẽ load `funasr.AutoModel(model="iic/emotion2vec_base")`.

Trên Kaggle:

1. Bật Internet.
2. Nếu chưa có package, notebook sẽ tự cài trên Kaggle. Có thể ép cài bằng `INSTALL_EMOTION2VEC_DEPS=1`.
3. Add Kaggle Dataset có `audio_wav/`, `metadata/`, `splits/`.

Nếu chỉ muốn test nhanh, set `MAX_FILES=50`, nhưng để train 03/04 nghiêm túc thì phải extract full 6,877 mẫu.

In [ ]:
model = load_emotion2vec_model()
sample_ids = []
e2v_rows = []
acoustic_rows = []
failures = []
acoustic_names = None

start = time.time()
for i, row in metadata.iterrows():
    sid = str(row["train_sample_id"])
    try:
        wav_path = resolve_wav_path(row)
        y, sr = load_audio_16k(wav_path)
        a_vec, a_names = acoustic_vector(y, sr)
        e_vec = extract_emotion2vec_embedding(model, wav_path)
        if acoustic_names is None:
            acoustic_names = a_names
        sample_ids.append(sid)
        acoustic_rows.append(a_vec)
        e2v_rows.append(e_vec)
    except Exception as exc:
        failures.append({"train_sample_id": sid, "error": repr(exc)})
        print("FAILED", sid, repr(exc))
        break
    if (i + 1) % 100 == 0:
        print(f"{i+1}/{len(metadata)} done, elapsed={time.time()-start:.1f}s")

if failures:
    pd.DataFrame(failures).to_csv(WORKING_FEATURE_DIR / "feature_extraction_failures.csv", index=False, encoding="utf-8-sig")
    raise RuntimeError("Feature extraction stopped because at least one sample failed. See feature_extraction_failures.csv")

max_e_dim = max(len(x) for x in e2v_rows)
e2v = np.zeros((len(e2v_rows), max_e_dim), dtype=np.float32)
for i, x in enumerate(e2v_rows):
    e2v[i, : len(x)] = x
acoustic = np.vstack(acoustic_rows).astype(np.float32)

np.savez_compressed(
    WORKING_FEATURE_CACHE_PATH,
    sample_ids=np.asarray(sample_ids, dtype=str),
    emotion2vec=e2v,
    acoustic=acoustic,
    acoustic_names=np.asarray(acoustic_names, dtype=str),
    model_name=np.asarray([EMOTION2VEC_MODEL_NAME], dtype=str),
)

print("Saved:", WORKING_FEATURE_CACHE_PATH)
print("emotion2vec:", e2v.shape)
print("acoustic:", acoustic.shape)

## 7. Kiểm tra cache sau khi extract

Cell cuối xác nhận file `.npz` đã được tạo đúng schema để notebook 03/04 có thể đọc ngay.

In [ ]:
if not WORKING_FEATURE_CACHE_PATH.exists():
    raise FileNotFoundError(f"Chưa thấy cache sau extraction: {WORKING_FEATURE_CACHE_PATH}")

cache_check = np.load(WORKING_FEATURE_CACHE_PATH, allow_pickle=True)
cache_summary = {
    "cache_path": str(WORKING_FEATURE_CACHE_PATH),
    "n_sample_ids": int(len(cache_check["sample_ids"])),
    "emotion2vec_shape": tuple(cache_check["emotion2vec"].shape),
    "acoustic_shape": tuple(cache_check["acoustic"].shape),
    "n_acoustic_names": int(len(cache_check["acoustic_names"])) if "acoustic_names" in cache_check else 0,
    "model_name": str(cache_check["model_name"][0]) if "model_name" in cache_check else "",
}
cache_summary_df = pd.DataFrame([cache_summary])
cache_summary_df.to_csv(WORKING_REPORT_DIR / "feature_cache_summary.csv", index=False, encoding="utf-8-sig")
display(cache_summary_df)

report_lines = [
    "# IEMOCAP full feature extraction report",
    "",
    f"- Cache path: `{WORKING_FEATURE_CACHE_PATH}`",
    f"- Samples: **{cache_summary['n_sample_ids']}**",
    f"- Emotion2Vec shape: **{cache_summary['emotion2vec_shape']}**",
    f"- Acoustic shape: **{cache_summary['acoustic_shape']}**",
    f"- Acoustic feature count: **{cache_summary['n_acoustic_names']}**",
    "",
    "Notebook 03/04 sẽ dùng cache này để train mô hình full hai nhánh: Emotion2Vec branch + acoustic branch.",
]
report_path = WORKING_REPORT_DIR / "feature_extraction_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
display(Markdown("\n".join(report_lines)))
print("Saved report:", report_path)

## 8. Zip toàn bộ output của notebook 02

Cell này gom các file do notebook 02 sinh ra để bạn tải về từ Kaggle:

- `features/iemocap_full_emotion2vec_acoustic_cache.npz`
- `feature_reports/`
- `feature_figures/`

Không zip `audio_wav/` vì đó là input raw audio rất lớn, không phải output của notebook.

In [ ]:
zip_path = Path("/kaggle/working/iemocap_full_feature_outputs.zip") if Path("/kaggle/working").exists() else WORKING_DATA_DIR.parent / "iemocap_full_feature_outputs.zip"
zip_folder(
    WORKING_DATA_DIR,
    zip_path,
    exclude_dir_names={"audio_wav"},
    exclude_suffixes=set(),
)
print("ZIP_OUTPUT:", zip_path)
print("ZIP_SIZE_MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))